# Week 5: Inheritance & Polymorphism
### PHASE 2: Composing Components

*📚 Object Oriented Programming · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

| # | Objective |
|---|---|
| 1 | Understand the **is-a** relationship and why inheritance exists |
| 2 | Create a **child class** that inherits from a **parent class** |
| 3 | Use `super()` to call parent class methods |
| 4 | **Override** methods in child classes |
| 5 | Add **new methods** unique to child classes |
| 6 | Understand **polymorphism** — same method name, different behavior |
| 7 | Know **when to use** inheritance (and when not to) |

## 🎯 Core Mastery Connection

When multiple components share the same interface, they become interchangeable in your composed system. Inheritance and polymorphism let you define a family of components — `TemperatureSensor`, `PressureSensor`, `HumiditySensor` — that all respond to the same `display()` call. A `MonitoringSystem` can work with *any* sensor type without knowing the specifics, because every sensor honors the shared interface.

---

## Part 1: What is Inheritance?

In engineering, we often have **general categories** and **specific types**:

| General (Parent) | Specific (Child) |
|---|---|
| Sensor | TemperatureSensor, PressureSensor |
| Motor | DCMotor, StepperMotor |
| Vehicle | Car, Truck |

A `TemperatureSensor` **is a** `Sensor`. It has everything a sensor has, plus some extra features.

This "is-a" relationship is what **inheritance** models in OOP.

**Figure 1.1** — Inheritance hierarchy

```
        Sensor
       /      \
  TempSensor  PressureSensor
```

> **Key idea:** The child class gets all the attributes and methods of the parent class automatically.

In [ ]:
# A simple parent class
class Sensor:
    def __init__(self, name, unit):
        self.name = name      # sensor name
        self.unit = unit      # measurement unit (e.g., "°C")
        self.value = 0.0      # current reading

    def read(self):
        """Return the current sensor value."""
        return self.value

    def display(self):
        """Print the sensor reading."""
        print(f"{self.name}: {self.value} {self.unit}")

# Create a generic sensor
s = Sensor("GenericSensor", "V")
s.value = 3.3
s.display()

---

## Part 2: Creating a Child Class

To create a child class, we put the parent class name in parentheses:

```python
class ChildClass(ParentClass):
    ...
```

The child class **inherits** all methods and attributes from the parent.

In [ ]:
# Child class inherits from Sensor
class TemperatureSensor(Sensor):
    pass  # no new code yet — it inherits everything

# Create a temperature sensor
ts = TemperatureSensor("TMP36", "°C")
ts.value = 22.5

# It already has all Sensor methods!
ts.display()
print(f"Reading: {ts.read()}")

**Figure 5.1** — Inheritance: child class inherits all parent methods

In [ ]:
# Check the relationship
print(isinstance(ts, TemperatureSensor))  # True — it IS a TemperatureSensor
print(isinstance(ts, Sensor))             # True — it IS ALSO a Sensor
print(type(ts))                           # <class 'TemperatureSensor'>

---

## Part 3: The `super()` Function

When a child class needs its **own** `__init__`, it should still call the parent's `__init__` to set up the inherited attributes.

We use `super()` to access the parent class:

```python
super().__init__(...)   # call the parent's __init__
super().method_name()   # call any parent method
```

**Figure 3.1** — How `super()` works

```
TemperatureSensor.__init__
    └── super().__init__(name, unit)  →  Sensor.__init__
    └── self.min_temp = min_temp      →  new attribute
```

In [ ]:
class TemperatureSensor(Sensor):
    def __init__(self, name, min_temp=-40, max_temp=125):
        # Call the parent's __init__ first
        super().__init__(name, "°C")
        # Then add child-specific attributes
        self.min_temp = min_temp  # minimum measurable temperature
        self.max_temp = max_temp  # maximum measurable temperature

# Create a temperature sensor with custom range
ts = TemperatureSensor("TMP36", min_temp=-40, max_temp=125)
ts.value = 22.5
ts.display()  # inherited from Sensor
print(f"Range: {ts.min_temp} to {ts.max_temp} °C")

**Figure 5.2** — Adding a second child class with super()

In [ ]:
# Another child class
class PressureSensor(Sensor):
    def __init__(self, name, max_pressure=1000):
        # Call parent __init__ with unit set to "kPa"
        super().__init__(name, "kPa")
        # Child-specific attribute
        self.max_pressure = max_pressure

# Create a pressure sensor
ps = PressureSensor("BMP280", max_pressure=1100)
ps.value = 101.3
ps.display()
print(f"Max pressure: {ps.max_pressure} kPa")

---

## Part 4: Method Overriding

A child class can **override** (replace) a parent method by defining a method with the **same name**.

| Term | Meaning |
|---|---|
| **Inherit** | Use the parent's method as-is |
| **Override** | Replace the parent's method with a new version |

This is useful when the child needs **different behavior**.

**Figure 5.3** — Method overriding: child replaces parent's display()

In [ ]:
class TemperatureSensor(Sensor):
    def __init__(self, name, min_temp=-40, max_temp=125):
        super().__init__(name, "°C")
        self.min_temp = min_temp
        self.max_temp = max_temp

    def display(self):
        """Override: show temperature with a warning if out of range."""
        status = "OK"
        # Check if value is outside the valid range
        if self.value < self.min_temp or self.value > self.max_temp:
            status = "OUT OF RANGE"
        print(f"{self.name}: {self.value} {self.unit} [{status}]")

# Test the overridden method
ts = TemperatureSensor("TMP36")
ts.value = 22.5
ts.display()  # uses the NEW display method

ts.value = 200.0  # too hot!
ts.display()  # shows OUT OF RANGE

**Figure 5.4** — Calling super() inside an overridden method

In [ ]:
# You can also call super() INSIDE an overridden method
class PressureSensor(Sensor):
    def __init__(self, name, max_pressure=1100):
        super().__init__(name, "kPa")
        self.max_pressure = max_pressure

    def display(self):
        """Override: call parent display, then add extra info."""
        super().display()  # call the original Sensor.display()
        # Add extra information
        percentage = (self.value / self.max_pressure) * 100
        print(f"  → {percentage:.1f}% of max capacity")

ps = PressureSensor("BMP280")
ps.value = 101.3
ps.display()

---

## Part 5: Adding New Methods to Child Classes

Child classes can have methods that the parent does **not** have.

These methods are **specific** to the child class only.

**Figure 5.5** — Adding child-only methods (to_fahrenheit, is_freezing)

In [ ]:
class TemperatureSensor(Sensor):
    def __init__(self, name, min_temp=-40, max_temp=125):
        super().__init__(name, "°C")
        self.min_temp = min_temp
        self.max_temp = max_temp

    def to_fahrenheit(self):
        """Convert current reading to Fahrenheit. Only for temperature!"""
        return self.value * 9 / 5 + 32

    def is_freezing(self):
        """Check if temperature is below 0°C."""
        return self.value <= 0

ts = TemperatureSensor("TMP36")
ts.value = -5.0
print(f"Celsius: {ts.value}")
print(f"Fahrenheit: {ts.to_fahrenheit()}")  # child-only method
print(f"Freezing: {ts.is_freezing()}")       # child-only method

In [ ]:
# A generic Sensor does NOT have to_fahrenheit()
s = Sensor("Generic", "V")
try:
    s.to_fahrenheit()  # this will cause an error
except AttributeError as e:
    print(f"Error: {e}")
    print("→ Only TemperatureSensor has to_fahrenheit()!")

---

## Part 6: Polymorphism — Same Interface, Different Behavior

**Polymorphism** means "many forms". In OOP:

> Different objects respond to the **same method call** in their **own way**.

This is powerful because you can write code that works with **any** sensor type without knowing the specific type.

**Figure 6.1** — Polymorphism in action

```
for sensor in sensor_list:
    sensor.display()    ← same method call
         │
         ├── TemperatureSensor.display()  → shows temp + range check
         └── PressureSensor.display()     → shows pressure + % capacity
```

In [ ]:
# Redefine both classes for a clean example
class Sensor:
    def __init__(self, name, unit):
        self.name = name
        self.unit = unit
        self.value = 0.0

    def display(self):
        print(f"{self.name}: {self.value} {self.unit}")

class TemperatureSensor(Sensor):
    def __init__(self, name):
        super().__init__(name, "°C")

    def display(self):
        # Show a warning emoji if hot
        icon = "🔥" if self.value > 50 else "🌡️"
        print(f"{icon} {self.name}: {self.value} {self.unit}")

class PressureSensor(Sensor):
    def __init__(self, name):
        super().__init__(name, "kPa")

    def display(self):
        # Show atmospheric context
        note = "(normal)" if 95 <= self.value <= 105 else "(unusual)"
        print(f"📊 {self.name}: {self.value} {self.unit} {note}")

**Figure 5.6** — Polymorphism with a shared interface

In [ ]:
# Polymorphism: one loop, different behaviors
sensors = [
    TemperatureSensor("TMP36"),
    PressureSensor("BMP280"),
    TemperatureSensor("DHT22"),
]

# Set some values
sensors[0].value = 22.5
sensors[1].value = 101.3
sensors[2].value = 75.0  # hot!

# Same method call, different output
print("=== Sensor Dashboard ===")
for sensor in sensors:
    sensor.display()  # each sensor displays in its own way

**Figure 5.7** — A polymorphic function that works with any sensor type

In [ ]:
# A function that works with ANY sensor — thanks to polymorphism
def print_report(sensor_list):
    """Print a report for any list of sensors."""
    print("\n--- Sensor Report ---")
    for s in sensor_list:
        s.display()  # we don't care what type it is
    print(f"Total sensors: {len(sensor_list)}")
    print("--- End Report ---\n")

print_report(sensors)

---

## Part 7: When to Use Inheritance (and When Not To)

### ✅ Use inheritance when:

| Situation | Example |
|---|---|
| There is a clear **is-a** relationship | A `DCMotor` **is a** `Motor` |
| Child classes share **most** of the parent's behavior | All sensors have `read()` and `display()` |
| You want **polymorphism** | Process any sensor the same way |

### ❌ Avoid inheritance when:

| Situation | Better approach |
|---|---|
| There is a **has-a** relationship | A `Robot` **has a** `Motor` → use composition |
| You only need one or two methods | Just define them directly |
| The hierarchy gets too deep (3+ levels) | Keep it simple |

> **Rule of thumb:** If you can say "X **is a** Y", inheritance might be right. If you say "X **has a** Y", use composition instead.

**Figure 5.8** — Is-a (inheritance) vs Has-a (composition)

In [ ]:
# ✅ Good: is-a relationship
class Motor:
    def __init__(self, name, max_rpm):
        self.name = name
        self.max_rpm = max_rpm

class DCMotor(Motor):  # a DCMotor IS A Motor
    def __init__(self, name, max_rpm, voltage):
        super().__init__(name, max_rpm)
        self.voltage = voltage

# ❌ Bad: has-a relationship — don't use inheritance here
# class Robot(Motor):  # a Robot is NOT a Motor!

# ✅ Good: use composition instead
class Robot:
    def __init__(self, name):
        self.name = name
        self.motor = DCMotor("main_motor", 3000, 12)  # Robot HAS A Motor

r = Robot("MechBot")
print(f"{r.name} uses motor: {r.motor.name} ({r.motor.voltage}V)")

---

## 🛠️ Studio: Sensor Subtypes

Let's build a small sensor monitoring system with inheritance and polymorphism.

**Figure 5.9** — Base Sensor class and TemperatureSensor subclass

In [ ]:
# Complete studio example
class Sensor:
    """Base sensor class for all sensor types."""
    def __init__(self, name, unit):
        self.name = name
        self.unit = unit
        self.readings = []  # store all readings

    def add_reading(self, value):
        """Add a new reading to the history."""
        self.readings.append(value)

    def average(self):
        """Calculate average of all readings."""
        if not self.readings:
            return 0.0
        return sum(self.readings) / len(self.readings)

    def display(self):
        """Show sensor info."""
        avg = self.average()
        print(f"{self.name} | Avg: {avg:.1f} {self.unit} | Readings: {len(self.readings)}")


class TemperatureSensor(Sensor):
    """Sensor that measures temperature in Celsius."""
    def __init__(self, name, warning_temp=50):
        super().__init__(name, "°C")        # always uses °C
        self.warning_temp = warning_temp     # threshold for warnings

    def display(self):
        """Show temperature with warning if average is high."""
        avg = self.average()
        warning = " ⚠️ HIGH" if avg > self.warning_temp else ""
        print(f"🌡️ {self.name} | Avg: {avg:.1f} {self.unit}{warning}")


class PressureSensor(Sensor):
    """Sensor that measures pressure in kPa."""
    def __init__(self, name, max_pressure=1100):
        super().__init__(name, "kPa")       # always uses kPa
        self.max_pressure = max_pressure

    def display(self):
        """Show pressure with percentage of max."""
        avg = self.average()
        pct = (avg / self.max_pressure) * 100
        print(f"📊 {self.name} | Avg: {avg:.1f} {self.unit} ({pct:.0f}% of max)")

**Figure 5.10** — Polymorphism with a shared interface

---

## 🎢 Exercises — Interchangeable Components

> **Composition connection:** Inheritance makes components *interchangeable*. When you build a `Dashboard` that calls `sensor.display()`, it does not matter whether the sensor is a `TemperatureSensor` or a `PressureSensor` — polymorphism handles it. This is how real composed systems stay flexible: swap one component for another without changing the system.

### Difficulty Guide

| Level | Meaning |
|---|---|
| 🟢 Easy | Apply what you learned directly |
| 🟡 Medium | Combine concepts or add small twists |
| 🔴 Challenge | Think deeper, design your own solution |

---

## 🎢 Exercises

### Difficulty Guide

| Level | Meaning |
|---|---|
| 🟢 Easy | Apply what you learned directly |
| 🟡 Medium | Combine concepts or add small twists |
| 🔴 Challenge | Think deeper, design your own solution |

### 🟢 Exercise 1 — Basic Inheritance

Create a `Vehicle` class with `name` and `max_speed` attributes and a `describe()` method.
Then create a `Car` class that inherits from `Vehicle` and adds a `num_doors` attribute.

<details><summary>💡 Hint</summary>

Use `super().__init__(name, max_speed)` in Car's `__init__` to set up the parent attributes.

</details>

In [ ]:
# ✏️ [EX1] Basic Inheritance
# Create Vehicle and Car classes below



### 🟢 Exercise 2 — Using super()

Create a `Component` class with `name` and `weight` (in grams).
Create a `Resistor` class that inherits from `Component` and adds `resistance` (in ohms).
Use `super()` to call the parent `__init__`.

<details><summary>💡 Hint</summary>

```python
class Resistor(Component):
    def __init__(self, name, weight, resistance):
        super().__init__(name, weight)
        self.resistance = resistance
```

</details>

In [ ]:
# ✏️ [EX2] Using super()



### 🟢 Exercise 3 — Method Overriding

Using the `Sensor` base class from the studio, create a `HumiditySensor` child class.
Override the `display()` method to show the humidity with a description: "Dry" (< 30), "Normal" (30–60), or "Humid" (> 60).

<details><summary>💡 Hint</summary>

Use `self.average()` (inherited!) to get the average, then use if/elif/else to pick the description.

</details>

In [ ]:
# ✏️ [EX3] Method Overriding



### 🟡 Exercise 4 — Adding Child Methods

Create a `Battery` class that inherits from `Component` (from EX2).
Add these child-only methods:
- `charge_time(current_mA)` → returns voltage * capacity / current_mA (hours)
- `is_low()` → returns True if voltage < 3.0

The Battery should have `voltage` and `capacity_mAh` as extra attributes.

<details><summary>💡 Hint</summary>

Call `super().__init__(name, weight)` then set `self.voltage` and `self.capacity_mAh`.

</details>

In [ ]:
# ✏️ [EX4] Adding Child Methods



### 🟡 Exercise 5 — Polymorphism with a Loop

Create three different sensor types (use `TemperatureSensor`, `PressureSensor`, and your `HumiditySensor` from EX3).
Put them all in a list, add some readings to each, and use a single loop to call `display()` on all of them.

<details><summary>💡 Hint</summary>

```python
sensors = [temp_sensor, pressure_sensor, humidity_sensor]
for s in sensors:
    s.display()
```

</details>

In [ ]:
# ✏️ [EX5] Polymorphism with a Loop



### 🟡 Exercise 6 — isinstance() Check

Write a function `count_by_type(sensor_list, sensor_type)` that counts how many sensors in a list are of a given type.

Example: `count_by_type(sensors, TemperatureSensor)` → `2`

<details><summary>💡 Hint</summary>

Use `isinstance(sensor, sensor_type)` inside a loop and count the matches.

</details>

In [ ]:
# ✏️ [EX6] isinstance() Check



### 🟡 Exercise 7 — super() in Overridden Method

Create a `LoggingSensor` child of `Sensor`. Override `add_reading()` so that it:
1. Calls the parent's `add_reading()` using `super()`
2. Also prints a log message: `"[LOG] {name}: recorded {value}"`

<details><summary>💡 Hint</summary>

```python
def add_reading(self, value):
    super().add_reading(value)
    print(f"[LOG] {self.name}: recorded {value}")
```

</details>

In [ ]:
# ✏️ [EX7] super() in Overridden Method



### 🟡 Exercise 8 — Motor Hierarchy

Create a `Motor` base class with `name` and `max_rpm`, and a `status()` method that prints the name and max RPM.

Create two child classes:
- `DCMotor` — adds `voltage`, overrides `status()` to also show voltage
- `StepperMotor` — adds `steps_per_rev`, overrides `status()` to also show steps per revolution

<details><summary>💡 Hint</summary>

Each child should call `super().__init__(name, max_rpm)` and `super().status()` inside their overrides.

</details>

In [ ]:
# ✏️ [EX8] Motor Hierarchy



### 🔴 Exercise 9 — Polymorphic Function

Write a function `find_max_reading(sensor_list)` that takes a list of sensors (any type) and returns the sensor with the highest single reading.

Use `max(sensor.readings)` to find each sensor's highest reading. The function should work with **any** sensor subclass.

<details><summary>💡 Hint</summary>

Loop through the list, track the sensor with the highest `max(sensor.readings)`. Handle the case where `readings` is empty.

</details>

In [ ]:
# ✏️ [EX9] Polymorphic Function



### 🔴 Exercise 10 — Is-a vs Has-a

Design a `RobotArm` class that:
- **Has a** `Motor` (composition — store it as an attribute)
- **Has a** `Sensor` (composition)
- Has a `move(degrees)` method that prints "Moving {degrees}° using {motor_name}"
- Has a `check()` method that calls the sensor's `display()`

Do **not** use inheritance for this — use composition.

<details><summary>💡 Hint</summary>

```python
class RobotArm:
    def __init__(self, motor, sensor):
        self.motor = motor    # has-a Motor
        self.sensor = sensor  # has-a Sensor
```

</details>

In [ ]:
# ✏️ [EX10] Is-a vs Has-a



### 🔴 Exercise 11 — Sensor Dashboard

Create a `Dashboard` class that:
- Stores a list of sensors (any type)
- Has an `add_sensor(sensor)` method
- Has a `show_all()` method that calls `display()` on each sensor
- Has a `summary()` method that prints the total number of sensors and the average reading across ALL sensors

Test with at least 3 different sensor types.

<details><summary>💡 Hint</summary>

For `summary()`, loop through all sensors, collect all their `average()` values, and compute the mean.

</details>

In [ ]:
# ✏️ [EX11] Sensor Dashboard



---

## 🌉 Bridge to Next Week

This week we learned how to **share behavior** between classes using inheritance and how **polymorphism** lets us write flexible code.

But there is a problem: how do we **guarantee** that every child class implements certain methods?

Right now, nothing stops someone from creating a sensor subclass that forgets to implement `display()`. The code would still run — using the parent's version — which might not be correct.

**Next week**, we will learn about **Abstract Classes and Interfaces** — a way to create a "contract" that says: *"Every child class MUST implement these methods."*

```python
# Sneak peek — Week 6
from abc import ABC, abstractmethod

class Sensor(ABC):  # abstract class
    @abstractmethod
    def display(self):  # MUST be implemented by all children
        pass
```

---
## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_05"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")